# 💻 IntelliCode-SL | Coding SLM Fine-Tuning
**Model:** `Qwen2.5-Coder-3B-Instruct-bnb-4bit`  
**Tasks:** `debug` | `generate` | `modify`

> Run cells top to bottom. GPU runtime required (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────
!pip install -q unsloth transformers datasets peft accelerate bitsandbytes trl
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────
import os, torch
from google.colab import drive
from huggingface_hub import login
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer
print("✅ Imports done | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────
drive.mount("/content/drive")

DRIVE_BASE   = "/content/drive/MyDrive/IntelliCode-SL"
ADAPTER_SAVE = f"{DRIVE_BASE}/adapters/coding_adapter"

DATASET_PATHS = {
    "debug"    : f"{DRIVE_BASE}/datasets/debug_dataset.json",
    "generate" : f"{DRIVE_BASE}/datasets/generation_dataset.json",
    "modify"   : f"{DRIVE_BASE}/datasets/modification_dataset.json",
}

os.makedirs(ADAPTER_SAVE, exist_ok=True)
print("✅ Drive mounted")
for k, v in DATASET_PATHS.items():
    print(f"   {k:10s} → {v}")
print(f"   adapter    → {ADAPTER_SAVE}")

In [ ]:
# ── Cell 4: HuggingFace Login ──────────────────────────────────
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← paste your token here
login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")

In [ ]:
# ── Cell 5: Load Model via Unsloth ─────────────────────────────
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
print("✅ Model loaded")

In [ ]:
# ── Cell 6: Attach LoRA Adapter ────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("✅ LoRA adapter attached")
model.print_trainable_parameters()

## 📁 Dataset Format
Each JSON file should have samples like:
```json
[
  {"input": "<buggy or base code>", "output": "<fixed/generated/modified code>"}
]
```
- `debug_dataset.json` → buggy code → fixed code  
- `generation_dataset.json` → description/prompt → code  
- `modification_dataset.json` → code + instruction → modified code

In [ ]:
# ── Cell 7: Load & Format Dataset ──────────────────────────────
TASK_PROMPTS = {
    "debug"    : "Fix the bug in the following code and return the corrected version.",
    "generate" : "Write code based on the following description.",
    "modify"   : "Modify the following code according to the given instruction.",
}

EOS = tokenizer.eos_token

PROMPT_TEMPLATE = """### Task: {task_instruction}

### Input:
{input}

### Output:
{output}"""

def format_sample(sample):
    task_instr = TASK_PROMPTS.get(sample.get("task","generate"), "Complete the coding task.")
    return {"text": PROMPT_TEMPLATE.format(
        task_instruction=task_instr,
        input=sample["input"],
        output=sample["output"]
    ) + EOS}

all_splits = []
for task_name, path in DATASET_PATHS.items():
    ds = load_dataset("json", data_files=path, split="train")
    ds = ds.map(lambda x, t=task_name: {**x, "task": t})
    all_splits.append(ds)
    print(f"  Loaded {len(ds):>4d} samples for '{task_name}'")

merged_dataset = concatenate_datasets(all_splits).shuffle(seed=42)
dataset        = merged_dataset.map(format_sample)
print(f"\n✅ Total samples: {len(dataset)}")

In [ ]:
# ── Cell 8: Training Arguments ─────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = "/content/coding_checkpoints",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    save_strategy               = "epoch",
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    report_to                   = "none",
)
print("✅ Training args set")

In [ ]:
# ── Cell 9: Train ──────────────────────────────────────────────
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args               = training_args,
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# ── Cell 10: Save Adapter to Google Drive ──────────────────────
model.save_pretrained(ADAPTER_SAVE)
tokenizer.save_pretrained(ADAPTER_SAVE)
print(f"✅ Adapter saved to Drive → {ADAPTER_SAVE}")

In [ ]:
# ── Cell 11: Quick Inference Test ──────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = PROMPT_TEMPLATE.format(
    task_instruction = TASK_PROMPTS["debug"],
    input = "def add(a, b):\n    return a - b   # this should add not subtract",
    output = ""
)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.2, do_sample=True)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Fixed Code:")
print(result.split("### Output:")[-1].strip())